# Transformer

In [1]:
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def one_hot(x, vocab):
    out = np.zeros((x.shape[0], x.shape[1], vocab))
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            out[i, j, x[i, j]] = 1.0
    return out


In [2]:
class SelfAttention:
    def __init__(self, d_model):
        self.Wq = np.random.randn(d_model, d_model) / np.sqrt(d_model)
        self.Wk = np.random.randn(d_model, d_model) / np.sqrt(d_model)
        self.Wv = np.random.randn(d_model, d_model) / np.sqrt(d_model)

    def forward(self, x):
        Q = x @ self.Wq
        K = x @ self.Wk
        V = x @ self.Wv

        scores = Q @ K.transpose(0, 2, 1) / np.sqrt(x.shape[-1])
        weights = softmax(scores, axis=-1)
        return weights @ V


In [3]:
class FeedForward:
    def __init__(self, d_model, d_hidden):
        self.W1 = np.random.randn(d_model, d_hidden) / np.sqrt(d_model)
        self.b1 = np.zeros(d_hidden)
        self.W2 = np.random.randn(d_hidden, d_model) / np.sqrt(d_hidden)
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        h = np.maximum(0, x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2


In [4]:
class TransformerBlock:
    def __init__(self, d_model, d_hidden):
        self.attn = SelfAttention(d_model)
        self.ff = FeedForward(d_model, d_hidden)

    def forward(self, x):
        x = layer_norm(x + self.attn.forward(x))
        x = layer_norm(x + self.ff.forward(x))
        return x


In [5]:
class Transformer:
    def __init__(self, vocab, seq_len, d_model=32, d_hidden=64):
        self.vocab = vocab
        self.seq_len = seq_len
        self.d_model = d_model

        self.token_embed = np.random.randn(vocab, d_model) / np.sqrt(d_model)
        self.pos_embed = np.random.randn(seq_len, d_model) / np.sqrt(d_model)

        self.block = TransformerBlock(d_model, d_hidden)

        self.Wo = np.random.randn(d_model, vocab) / np.sqrt(d_model)
        self.bo = np.zeros(vocab)

    def forward(self, x):
        B, L = x.shape
        h = self.token_embed[x] + self.pos_embed[np.newaxis, :, :]
        h = self.block.forward(h)
        logits = h @ self.Wo + self.bo
        return logits


In [6]:
def generate_batch(batch_size, seq_len, vocab):
    x = np.random.randint(0, vocab, size=(batch_size, seq_len))
    y = np.sort(x, axis=1)
    return x, y

In [7]:
def cross_entropy(logits, targets):
    probs = softmax(logits, axis=-1)
    loss = 0.0
    B, L = targets.shape
    for i in range(B):
        for j in range(L):
            loss -= np.log(probs[i, j, targets[i, j]] + 1e-9)
    return loss / (B * L)


In [12]:
def train(model, steps=10000, lr=0.1):
    for step in range(steps):
        x, y = generate_batch(16, model.seq_len, model.vocab)
        logits = model.forward(x)
        loss = cross_entropy(logits, y)

        # Gradient on output projection only (toy training)
        probs = softmax(logits, axis=-1)
        grad = probs
        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                grad[i, j, y[i, j]] -= 1

        grad /= (x.shape[0] * x.shape[1])

        dWo = np.zeros_like(model.Wo)
        dbo = grad.sum(axis=(0, 1))

        h = model.token_embed[x] + model.pos_embed[np.newaxis, :, :]
        h = model.block.forward(h)

        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                dWo += np.outer(h[i, j], grad[i, j])

        model.Wo -= lr * dWo
        model.bo -= lr * dbo

        if step % 50 == 0:
            print(f"step {step} | loss {loss:.4f}")


In [13]:
model = Transformer(vocab=10, seq_len=5)
train(model)

step 0 | loss 2.7499
step 50 | loss 1.7616
step 100 | loss 1.7013
step 150 | loss 1.7212
step 200 | loss 1.6030
step 250 | loss 1.4849
step 300 | loss 1.3902
step 350 | loss 1.4961
step 400 | loss 1.2924
step 450 | loss 1.3772
step 500 | loss 1.3489
step 550 | loss 1.3784
step 600 | loss 1.2090
step 650 | loss 1.1962
step 700 | loss 1.2961
step 750 | loss 1.2685
step 800 | loss 1.2342
step 850 | loss 1.2563
step 900 | loss 1.2532
step 950 | loss 1.1632
step 1000 | loss 1.1729
step 1050 | loss 1.0900
step 1100 | loss 1.0623
step 1150 | loss 1.2813
step 1200 | loss 0.9710
step 1250 | loss 1.0624
step 1300 | loss 1.1183
step 1350 | loss 1.1100
step 1400 | loss 1.0559
step 1450 | loss 1.1231
step 1500 | loss 1.1790
step 1550 | loss 0.9785
step 1600 | loss 0.9700
step 1650 | loss 1.1821
step 1700 | loss 1.0162
step 1750 | loss 1.0478
step 1800 | loss 0.9209
step 1850 | loss 1.0970
step 1900 | loss 1.0764
step 1950 | loss 1.0747
step 2000 | loss 0.9265
step 2050 | loss 1.1745
step 2100 | los

In [14]:
x, y = generate_batch(1, 5, 10)
pred = model.forward(x).argmax(axis=-1)

print("input :", x)
print("target:", y)
print("pred  :", pred)


input : [[6 6 3 3 8]]
target: [[3 3 6 6 8]]
pred  : [[3 3 3 6 9]]


## References

- [Transformer](https://righteous-guardian-68f.notion.site/Transformer-3f35bea9676546abbc588911bfd83c41?source=copy_link)